In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('data/Order_Details.csv')

In [ ]:
order = pd.read_csv('data/Orders.csv')

In [ ]:
data = pd.merge(data, order[['ORDERID', 'DATE_']], how='left', on='ORDERID')

In [ ]:
data = data[data['ITEMCODE'] == 9045]

In [ ]:
data_grop = data.groupby(['ITEMCODE', 'DATE_']).agg(
    {
        'AMOUNT': 'sum',
        'UNITPRICE': 'mean',
        'ORDERID': 'nunique'

    }
).reset_index()

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t

# =========================================================
# ДАННЫЕ
# =========================================================

train = data_grop[['UNITPRICE', 'month', 'dayofweek', 'day', 'year', 'quarter']]
target = data_grop['AMOUNT']

# =========================================================
# ОБУЧЕНИЕ
# =========================================================

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train)

poly = PolynomialFeatures(degree=2, include_bias=False)
train_poly = poly.fit_transform(train_scaled)

model = Ridge()
model.fit(train_poly, target)

print(model.coef_)

pred = model.predict(train_poly)

print("R2:", r2_score(target, pred))
print("MSE:", np.mean((target - pred)**2))

# =========================================================
# PREDICTION INTERVAL
# =========================================================

X = train_poly
y = target.values

n = len(y)
p = X.shape[1]

# Остатки
residuals = y - pred

# Оценка дисперсии ошибок
mse = np.sum(residuals**2) / (n - p)
s_err = np.sqrt(mse)

# leverage
XtX_inv = np.linalg.inv(X.T @ X)
h = np.sum(X @ XtX_inv * X, axis=1)

# t-value
alpha = 0.05
t_value = t.ppf(1 - alpha/2, df=n - p)

# Prediction interval
pi = t_value * s_err * np.sqrt(1 + h)

lower = pred - pi
upper = pred + pi

# =========================================================
# ГРАФИК
# =========================================================

plot_df = data_grop.copy()

plot_df['pred'] = pred
plot_df['lower'] = lower
plot_df['upper'] = upper

plot_df = plot_df.sort_values('UNITPRICE')

fig, ax = plt.subplots(figsize=(5, 5))

sns.scatterplot(
    data=plot_df,
    x='UNITPRICE',
    y='AMOUNT',
    alpha=0.5,
    label='Фактические данные',
    ax=ax
)

sns.lineplot(
    data=plot_df,
    x='UNITPRICE',
    y='pred',
    color='orange',
    linewidth=2,
    label='Прогноз',
    ax=ax
)

ax.fill_between(
    plot_df['UNITPRICE'],
    plot_df['lower'],
    plot_df['upper'],
    alpha=0.25,
    label='95% Prediction Interval'
)

ax.set_title('Зависимость количества продаж от цены')
ax.set_xlabel('Цена')
ax.set_ylabel('Количество продаж')

plt.grid(True)
plt.legend()

plt.show()
